# Machine Translation for Public Service Announcements (PSAs)

## Week 1: Data Collection

### Group 10

---

## Project Description

This notebook implements the data collection phase of an English-to-Dholuo Neural Machine Translation (NMT) project for Public Service Announcements (PSAs).

Public Service Announcements are official messages issued by government agencies and humanitarian organizations to educate, inform, or warn the public about issues such as health, education, agriculture, governance, and public safety.

The purpose of this notebook is to collect authentic English PSA sentences from trusted organizations, clean the collected data, and prepare it for translation into Dholuo. The resulting bilingual corpus will later be used to train and evaluate a machine translation model.

---

## Expected Output

At the end of this notebook, the project will produce:

- A clean English PSA dataset
- Sentence-level records
- Metadata describing each record
- CSV files stored in the project data directory

These outputs will be used during preprocessing and model training.

# Objectives

The objectives of this notebook are to:

1. Create the required project folder structure.
2. Import all required Python libraries.
3. Register trusted Public Service Announcement data sources.
4. Build reusable data collection tools.
5. Collect English PSA text.
6. Extract individual sentences.
7. Remove duplicate and noisy records.
8. Save a clean English PSA dataset for translation into Dholuo.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np

import requests

from bs4 import BeautifulSoup

import nltk

from nltk.tokenize import sent_tokenize

nltk.download("punkt")

print("Libraries imported successfully.")

Libraries imported successfully.


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\Levin\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!


# Create Project Folder Structure

A standard project directory structure improves reproducibility and makes the project easier to maintain.

Folders are created automatically if they do not already exist.

In [ ]:
folders = [

    "data/raw",

    "data/interim",

    "data/processed",

    "data/external",

    "Models",

    "Reports"

]

for folder in folders:

    Path(folder).mkdir(parents=True, exist_ok=True)

print("Project folders created successfully.")

Project folders created successfully.


# Register Trusted PSA Sources

Only trusted public organizations are used as data sources.

These organizations publish verified Public Service Announcements that are appropriate for building a machine translation corpus.

In [ ]:
sources = pd.DataFrame({

    "Source":[

        "World Health Organization",

        "Kenya Ministry of Health",

        "Kenya Red Cross",

        "Ministry of Education",

        "Ministry of Agriculture",

        "National Transport and Safety Authority",

        "National Police Service"

    ],

    "Domain":[

        "Health",

        "Health",

        "Health",

        "Education",

        "Agriculture",

        "Road Safety",

        "Security"

    ],

    "Website":[

        "https://www.who.int",

        "https://www.health.go.ke",

        "https://www.redcross.or.ke",

        "https://www.education.go.ke",

        "https://www.kilimo.go.ke",

        "https://ntsa.go.ke",

        "https://www.nationalpolice.go.ke"

    ]

})

sources

,Source,Domain,Website
0,World Health Organization,Health,https://www.who.int
1,Kenya Ministry of Health,Health,https://www.health.go.ke
2,Kenya Red Cross,Health,https://www.redcross.or.ke
3,Ministry of Education,Education,https://www.education.go.ke
4,Ministry of Agriculture,Agriculture,https://www.kilimo.go.ke
5,National Transport and Safety Authority,Road Safety,https://ntsa.go.ke
6,National Police Service,Security,https://www.nationalpolice.go.ke


# Save Source Registry

The registered sources are saved as a CSV file.

This makes the project reproducible because the same data sources can be reused during future data collection.

In [ ]:
sources.to_csv(

    "data/external/psa_sources.csv",

    index=False

)

print("Source registry saved successfully.")

sources

Source registry saved successfully.


,Source,Domain,Website
0,World Health Organization,Health,https://www.who.int
1,Kenya Ministry of Health,Health,https://www.health.go.ke
2,Kenya Red Cross,Health,https://www.redcross.or.ke
3,Ministry of Education,Education,https://www.education.go.ke
4,Ministry of Agriculture,Agriculture,https://www.kilimo.go.ke
5,National Transport and Safety Authority,Road Safety,https://ntsa.go.ke
6,National Police Service,Security,https://www.nationalpolice.go.ke


# Building a Reusable Web Scraper

Instead of creating a different scraper for each website, a reusable scraper is developed.

The scraper is responsible for:

- Downloading webpage content.
- Extracting paragraph text.
- Returning clean textual data for sentence extraction.

This approach improves code reuse and simplifies data collection from multiple trusted PSA sources.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from SRC.scraper import PSAScraper

scraper = PSAScraper()

print("Scraper initialized successfully.")

# Define PSA Pages

Rather than scraping website homepages, the project targets specific public advisory pages that contain instructional and awareness content.

These pages provide higher-quality Public Service Announcement text for the translation corpus.

In [ ]:
psa_pages = [

    {
        "source":"WHO",
        "domain":"Health",
        "url":"https://www.who.int/health-topics"
    },

    {
        "source":"WHO",
        "domain":"Health",
        "url":"https://www.who.int/news-room"
    }

]

psa_pages

In [ ]:
all_paragraphs = []

for page in psa_pages:

    print(f"Collecting from {page['source']}...")

    try:

        paragraphs = scraper.get_paragraphs(page["url"])

        for paragraph in paragraphs:

            all_paragraphs.append({

                "Source":page["source"],

                "Domain":page["domain"],

                "Paragraph":paragraph

            })

        print(f"Collected {len(paragraphs)} paragraphs")

    except Exception as e:

        print(e)

In [ ]:
paragraph_df = pd.DataFrame(all_paragraphs)

paragraph_df.head()

# Sentence Extraction

Machine translation models learn from aligned sentences rather than complete paragraphs.

The collected paragraphs are therefore divided into individual sentences using Natural Language Toolkit (NLTK).

In [ ]:
records = []

for _, row in paragraph_df.iterrows():

    sentences = sent_tokenize(row["Paragraph"])

    for sentence in sentences:

        if len(sentence) >= 40:

            records.append({

                "Source":row["Source"],

                "Domain":row["Domain"],

                "English":sentence

            })

english_df = pd.DataFrame(records)

english_df.head()

In [ ]:
print(f"Total collected sentences: {len(english_df)}")

english_df.sample(20)